# नमूना ०२: OpenAI SDK एकीकरण

यो नोटबुकले OpenAI Python SDK सँग उन्नत एकीकरण प्रदर्शन गर्दछ, जसले Microsoft Foundry Local र Azure OpenAI दुवैलाई समर्थन गर्दछ, स्ट्रिमिङ प्रतिक्रियाहरू र उचित त्रुटि व्यवस्थापन सहित।

## अवलोकन

यो नमूनाले निम्न कुराहरू प्रदर्शन गर्दछ:
- Foundry Local र Azure OpenAI बीच सहज स्विचिङ
- राम्रो प्रयोगकर्ता अनुभवका लागि स्ट्रिमिङ च्याट पूरा गर्ने
- FoundryLocalManager SDK को उचित प्रयोग
- बलियो त्रुटि व्यवस्थापन र फलब्याक संयन्त्रहरू
- उत्पादन-तयार कोड ढाँचाहरू


## आवश्यकताहरू

- **Foundry Local**: स्थापना गरिएको र चलिरहेको (स्थानीय अनुमानको लागि)
- **Python**: 3.8 वा पछिल्लो संस्करण OpenAI SDK सहित
- **Azure OpenAI**: मान्य अन्त बिन्दु र API कुञ्जी (क्लाउड अनुमानको लागि)

### निर्भरता स्थापना गर्नुहोस्


In [ ]:
# Install required packages
!pip install openai foundry-local-sdk

## पुस्तकालयहरू आयात गर्नुहोस् र सेटअप गर्नुहोस्


In [ ]:
import os
import sys
from openai import OpenAI
import time
from typing import Tuple

try:
    from foundry_local import FoundryLocalManager
    FOUNDRY_SDK_AVAILABLE = True
    print("✅ Foundry Local SDK is available")
except ImportError:
    FOUNDRY_SDK_AVAILABLE = False
    print("⚠️ Foundry Local SDK not available, manual configuration will be used")

## कन्फिगरेसन विकल्पहरू

Azure OpenAI (क्लाउड) वा Foundry Local (डिभाइसमा) बीच चयन गर्न उपयुक्त वातावरण भेरिएबलहरू सेट गर्नुहोस्।


### विकल्प १: Azure OpenAI कन्फिगरेसन

आफ्नो Azure OpenAI प्रमाणपत्रहरू अनकमेण्ट गरेर सेट गर्नुहोस्:


In [ ]:
# Azure OpenAI Configuration
# Uncomment and set your actual values

# os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-resource.openai.azure.com"
# os.environ["AZURE_OPENAI_API_KEY"] = "your-api-key-here"
# os.environ["AZURE_OPENAI_API_VERSION"] = "2024-08-01-preview"
# os.environ["MODEL"] = "your-deployment-name"  # e.g., "gpt-4"

print("Azure OpenAI configuration ready (if credentials are set)")

### विकल्प २: फाउन्ड्री स्थानीय कन्फिगरेसन

स्थानीय अनुमानका लागि डिफल्ट सेटिङहरू:


In [ ]:
# Foundry Local Configuration (default)
FOUNDRY_MODEL = "phi-4-mini"  # Change to your preferred model
FOUNDRY_BASE_URL = "http://localhost:51211"
FOUNDRY_API_KEY = ""  # Usually empty for local

print(f"Foundry Local configuration ready with model: {FOUNDRY_MODEL}")

## क्लाइन्ट फ्याक्टरी फङ्क्सनहरू

यी फङ्क्सनहरूले तपाईंको कन्फिगरेसनको आधारमा उपयुक्त OpenAI क्लाइन्ट सिर्जना गर्छन्:


In [ ]:
def create_azure_client() -> Tuple[OpenAI, str]:
    """Create Azure OpenAI client."""
    azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
    azure_api_key = os.environ.get("AZURE_OPENAI_API_KEY")
    azure_api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
    
    if not azure_endpoint or not azure_api_key:
        raise ValueError("Azure OpenAI endpoint and API key are required")
    
    model = os.environ.get("MODEL", "your-deployment-name")
    client = OpenAI(
        base_url=f"{azure_endpoint}/openai",
        api_key=azure_api_key,
        default_query={"api-version": azure_api_version},
    )
    
    print(f"🌐 Azure OpenAI client created with model: {model}")
    return client, model


def create_foundry_client() -> Tuple[OpenAI, str]:
    """Create Foundry Local client with SDK management."""
    alias = FOUNDRY_MODEL
    
    if FOUNDRY_SDK_AVAILABLE:
        try:
            # Use FoundryLocalManager for proper service management
            print(f"🔄 Initializing Foundry Local with model: {alias}...")
            manager = FoundryLocalManager(alias)
            model_info = manager.get_model_info(alias)
            
            # Configure OpenAI client to use local Foundry service
            client = OpenAI(
                base_url=manager.endpoint,
                api_key=manager.api_key  # API key is not required for local usage
            )
            
            print(f"✅ Foundry Local SDK initialized")
            print(f"   Endpoint: {manager.endpoint}")
            print(f"   Model: {model_info.id}")
            return client, model_info.id
        except Exception as e:
            print(f"⚠️ Could not use Foundry SDK ({e}), falling back to manual configuration")
    
    # Fallback to manual configuration
    client = OpenAI(
        base_url=f"{FOUNDRY_BASE_URL}/v1",
        api_key=FOUNDRY_API_KEY
    )
    
    print(f"🔧 Manual Foundry Local configuration")
    print(f"   Endpoint: {FOUNDRY_BASE_URL}/v1")
    print(f"   Model: {alias}")
    return client, alias

## क्लाइन्ट सुरु गर्नुहोस्

यसले स्वतः पत्ता लगाउँछ कि Azure OpenAI वा Foundry Local प्रयोग गर्ने:


In [ ]:
def initialize_client() -> Tuple[OpenAI, str, str]:
    """Initialize the appropriate OpenAI client."""
    
    # Check for Azure OpenAI configuration
    azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
    azure_api_key = os.environ.get("AZURE_OPENAI_API_KEY")
    
    if azure_endpoint and azure_api_key:
        print("🌐 Azure OpenAI configuration detected")
        try:
            client, model = create_azure_client()
            return client, model, "azure"
        except Exception as e:
            print(f"❌ Azure OpenAI initialization failed: {e}")
            print("🔄 Falling back to Foundry Local...")
    
    # Use Foundry Local
    print("🏠 Using Foundry Local configuration")
    try:
        client, model = create_foundry_client()
        return client, model, "foundry"
    except Exception as e:
        print(f"❌ Foundry Local initialization failed: {e}")
        raise

# Initialize the client
print("Initializing OpenAI client...")
print("=" * 50)
client, model, provider = initialize_client()
print("=" * 50)
print(f"✅ Initialization complete! Using {provider} with model: {model}")

## साधारण च्याट पूरा

साधारण च्याट पूरा परीक्षण गर्नुहोस्:


In [ ]:
def simple_chat(prompt: str, max_tokens: int = 150) -> str:
    """Send a simple chat message and get response."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

# Test basic chat
test_prompt = "Say hello from the SDK quickstart and explain what you are in one sentence."

print(f"👤 User: {test_prompt}")
print("\n🤖 Assistant:")
response = simple_chat(test_prompt)
print(response)

## स्ट्रिमिङ च्याट कम्प्लिसन

उपयोगकर्ताको अनुभव सुधार गर्न स्ट्रिमिङ प्रतिक्रिया प्रदर्शन गर्नुहोस्:


In [ ]:
def streaming_chat(prompt: str, max_tokens: int = 300) -> str:
    """Send a chat message with streaming response."""
    try:
        print("🤖 Assistant (streaming):")
        
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            stream=True
        )
        
        full_response = ""
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                content = chunk.choices[0].delta.content
                print(content, end="", flush=True)
                full_response += content
        
        print("\n")  # New line after streaming
        return full_response
    except Exception as e:
        error_msg = f"Error: {e}"
        print(error_msg)
        return error_msg

# Test streaming chat
streaming_prompt = "Explain the key benefits of using Microsoft Foundry Local for AI development. Include aspects like privacy, performance, and cost."

print(f"👤 User: {streaming_prompt}\n")
streaming_response = streaming_chat(streaming_prompt)

## बहु-पटकको संवाद

संवादको सन्दर्भलाई कायम राख्ने तरिका देखाउनुहोस्:


In [ ]:
class ConversationManager:
    """Manages multi-turn conversations with context."""
    
    def __init__(self, system_prompt: str = None):
        self.messages = []
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})
    
    def send_message(self, user_message: str, max_tokens: int = 200) -> str:
        """Send a message and get response while maintaining context."""
        # Add user message to conversation
        self.messages.append({"role": "user", "content": user_message})
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=self.messages,
                max_tokens=max_tokens
            )
            
            assistant_message = response.choices[0].message.content
            
            # Add assistant response to conversation
            self.messages.append({"role": "assistant", "content": assistant_message})
            
            return assistant_message
        except Exception as e:
            return f"Error: {e}"
    
    def get_conversation_length(self) -> int:
        """Get the number of messages in the conversation."""
        return len(self.messages)

# Create conversation manager with system prompt
system_prompt = "You are a helpful AI assistant specialized in explaining AI and machine learning concepts. Be concise but informative."
conversation = ConversationManager(system_prompt)

# Multi-turn conversation example
conversation_turns = [
    "What is the difference between AI inference on-device vs in the cloud?",
    "Which approach is better for privacy?",
    "What about performance and latency considerations?"
]

for i, turn in enumerate(conversation_turns, 1):
    print(f"\n{'='*60}")
    print(f"Turn {i}")
    print(f"{'='*60}")
    print(f"👤 User: {turn}")
    
    response = conversation.send_message(turn)
    print(f"\n🤖 Assistant: {response}")

print(f"\n📊 Conversation summary: {conversation.get_conversation_length()} messages total")

## प्रदर्शन तुलना

विभिन्न परिस्थितिहरूको लागि प्रतिक्रिया समयहरूको तुलना गर्नुहोस्:


In [ ]:
def benchmark_response_time(prompt: str, iterations: int = 3) -> dict:
    """Benchmark response time for a given prompt."""
    times = []
    responses = []
    
    for i in range(iterations):
        start_time = time.time()
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=50  # Keep responses short for timing
            )
            
            end_time = time.time()
            response_time = end_time - start_time
            
            times.append(response_time)
            responses.append(response.choices[0].message.content)
            
        except Exception as e:
            print(f"Error in iteration {i+1}: {e}")
    
    if times:
        avg_time = sum(times) / len(times)
        min_time = min(times)
        max_time = max(times)
        
        return {
            "average_time": avg_time,
            "min_time": min_time,
            "max_time": max_time,
            "all_times": times,
            "sample_response": responses[0] if responses else None
        }
    
    return {"error": "No successful responses"}

# Benchmark different types of prompts
benchmark_prompts = [
    "What is AI?",
    "Explain machine learning in simple terms.",
    "List 3 benefits of edge computing."
]

print(f"⏱️  Performance Benchmark ({provider} - {model})")
print("=" * 60)

for prompt in benchmark_prompts:
    print(f"\n📝 Prompt: '{prompt}'")
    results = benchmark_response_time(prompt)
    
    if "error" not in results:
        print(f"   ⏰ Average time: {results['average_time']:.2f}s")
        print(f"   ⚡ Fastest: {results['min_time']:.2f}s")
        print(f"   🐌 Slowest: {results['max_time']:.2f}s")
        print(f"   📄 Sample response: {results['sample_response'][:100]}...")
    else:
        print(f"   ❌ {results['error']}")

## उन्नत कन्फिगरेसन र त्रुटि व्यवस्थापन

विभिन्न प्यारामिटरहरू र त्रुटि परिदृश्यहरू परीक्षण गर्नुहोस्:


In [ ]:
def test_different_parameters():
    """Test chat completions with different parameters."""
    prompt = "Write a creative short story about AI."
    
    # Test different temperature values
    temperatures = [0.1, 0.5, 0.9]
    
    for temp in temperatures:
        print(f"\n🌡️ Temperature: {temp}")
        print("-" * 30)
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100,
                temperature=temp
            )
            
            print(f"Response: {response.choices[0].message.content[:150]}...")
            
        except Exception as e:
            print(f"Error with temperature {temp}: {e}")

test_different_parameters()

## सेवा स्वास्थ्य जाँच

व्यापक सेवा स्वास्थ्य र क्षमता जाँच:


In [ ]:
def comprehensive_health_check():
    """Perform comprehensive health check of the service."""
    print("🏥 Comprehensive Health Check")
    print("=" * 50)
    
    # 1. Check model listing
    try:
        models_response = client.models.list()
        available_models = [m.id for m in models_response.data]
        print(f"✅ Model listing: SUCCESS")
        print(f"   📋 Available models: {available_models}")
        
        if model in available_models:
            print(f"   ✅ Current model '{model}' is available")
        else:
            print(f"   ⚠️ Current model '{model}' not found in available models")
    except Exception as e:
        print(f"❌ Model listing: FAILED - {e}")
    
    # 2. Test basic completion
    try:
        test_response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Say 'Health check successful'"}],
            max_tokens=10
        )
        print(f"✅ Basic completion: SUCCESS")
        print(f"   💬 Response: {test_response.choices[0].message.content}")
    except Exception as e:
        print(f"❌ Basic completion: FAILED - {e}")
    
    # 3. Test streaming
    try:
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Count to 3"}],
            max_tokens=20,
            stream=True
        )
        
        stream_content = ""
        chunk_count = 0
        for chunk in stream:
            if chunk.choices[0].delta.content:
                stream_content += chunk.choices[0].delta.content
                chunk_count += 1
        
        print(f"✅ Streaming: SUCCESS")
        print(f"   📦 Chunks received: {chunk_count}")
        print(f"   💬 Streamed content: {stream_content.strip()}")
    except Exception as e:
        print(f"❌ Streaming: FAILED - {e}")
    
    # 4. Provider-specific information
    print(f"\n📊 Configuration Summary:")
    print(f"   🏢 Provider: {provider}")
    print(f"   🤖 Model: {model}")
    if provider == "foundry":
        print(f"   🏠 Foundry SDK Available: {FOUNDRY_SDK_AVAILABLE}")
        print(f"   🔗 Base URL: {FOUNDRY_BASE_URL}")
    elif provider == "azure":
        print(f"   🌐 Azure Endpoint: {os.environ.get('AZURE_OPENAI_ENDPOINT', 'Not set')}")
        print(f"   🔑 API Version: {os.environ.get('AZURE_OPENAI_API_VERSION', 'Not set')}")

comprehensive_health_check()

## अन्तरक्रियात्मक परीक्षण

यस सेल प्रयोग गरेर आफ्नै प्रम्प्टहरू अन्तरक्रियात्मक रूपमा परीक्षण गर्नुहोस्:


In [ ]:
# Interactive testing - modify the prompt below
custom_prompt = "Explain the concept of 'edge AI' and why it's becoming important."
use_streaming = True  # Set to False for regular completion

print(f"👤 Custom Prompt: {custom_prompt}\n")

if use_streaming:
    custom_response = streaming_chat(custom_prompt, max_tokens=250)
else:
    custom_response = simple_chat(custom_prompt, max_tokens=250)
    print(f"🤖 Assistant: {custom_response}")

## सारांश र आगामी कदमहरू

यस नोटबुकले OpenAI SDK को उन्नत एकीकरण प्रदर्शन गर्‍यो:

### ✅ समेटिएका मुख्य विशेषताहरू

1. **मल्टि-प्रोभाइडर समर्थन**: Azure OpenAI र Foundry Local बीच सहज स्विचिंग
2. **स्ट्रीमिङ प्रतिक्रियाहरू**: राम्रो UX का लागि वास्तविक-समय टोकन उत्पादन
3. **वार्तालाप व्यवस्थापन**: सन्दर्भसहित बहु-चरण वार्तालापहरू
4. **प्रदर्शन बेंचमार्किङ**: प्रतिक्रिया समयको मापन र विश्लेषण
5. **व्यापक स्वास्थ्य जाँचहरू**: सेवा प्रमाणीकरण र डायग्नोस्टिक्स
6. **त्रुटि व्यवस्थापन**: बलियो त्रुटि व्यवस्थापन र फलब्याक संयन्त्रहरू

### 🏆 Foundry Local बनाम Azure OpenAI

| पक्ष | Foundry Local | Azure OpenAI |
|------|---------------|--------------|
| **गोपनीयता** | ✅ डेटा स्थानीय रहन्छ | ⚠️ डेटा क्लाउडमा पठाइन्छ |
| **ढिलाइ** | ✅ कम (स्थानीय इन्फरेन्स) | ⚠️ बढी (नेटवर्क निर्भरता) |
| **खर्च** | ✅ निःशुल्क (हार्डवेयर पछि) | 💰 प्रति टोकन तिर्नु पर्ने |
| **अफलाइन** | ✅ अफलाइन काम गर्छ | ❌ इन्टरनेट आवश्यक |
| **मोडेल विविधता** | ⚠️ सीमित चयन | ✅ पूर्ण मोडेल पहुँच |
| **स्केलिङ** | ⚠️ हार्डवेयर निर्भर | ✅ असीमित स्केलिङ |

### 🚀 आगामी कदमहरू

- **नमूना 04**: Chainlit च्याट एप्लिकेसन निर्माण
- **नमूना 05**: मल्टि-एजेन्ट अर्केस्ट्रेसन प्रणालीहरू
- **नमूना 06**: बौद्धिक मोडेल रुटिङ
- **उत्पादन परिनियोजन**: स्केलिङ र निगरानीका विचारहरू

### 💡 उत्कृष्ट अभ्यासहरू

1. **प्रोभाइडरहरू बीच सधैं फलब्याक संयन्त्र लागू गर्नुहोस्**
2. **लामो प्रतिक्रियाहरूका लागि स्ट्रीमिङ प्रयोग गर्नुहोस्** ताकि प्रदर्शन राम्रो देखियोस्
3. **उत्पादन एप्लिकेसनहरूका लागि उचित त्रुटि व्यवस्थापन लागू गर्नुहोस्**
4. **विभिन्न प्रोभाइडरहरूको प्रतिक्रिया समय र खर्च निगरानी गर्नुहोस्**
5. **आफ्नो विशिष्ट आवश्यकताका आधारमा सही प्रोभाइडर चयन गर्नुहोस्**
